In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("figures", exist_ok=True)

def quat_normalize(q):
    return q / np.linalg.norm(q)

def quat_multiply(q1, q2):
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2

    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ])

def quat_to_dcm(q):
    q = quat_normalize(q)
    w, x, y, z = q

    return np.array([
        [1-2*(y**2+z**2), 2*(x*y - z*w), 2*(x*z + y*w)],
        [2*(x*y + z*w), 1-2*(x**2+z**2), 2*(y*z - x*w)],
        [2*(x*z - y*w), 2*(y*z + x*w), 1-2*(x**2+y**2)]
    ])

def small_angle_quat(dtheta):
    dq = np.zeros(4)
    dq[0] = 1.0
    dq[1:] = 0.5 * dtheta
    return quat_normalize(dq)

Quaternion + Rotation

In [ ]:
def omega_dot(omega, J):
    return np.linalg.inv(J) @ (-np.cross(omega, J @ omega))

def quat_dot(q, omega):
    wx, wy, wz = omega
    Omega = np.array([
        [0, -wx, -wy, -wz],
        [wx, 0, wz, -wy],
        [wy, -wz, 0, wx],
        [wz, wy, -wx, 0]
    ])
    return 0.5 * Omega @ q

def rk4_step(q, omega, dt, J):
    k1_q = quat_dot(q, omega)
    k1_w = omega_dot(omega, J)

    k2_q = quat_dot(q + 0.5*dt*k1_q, omega + 0.5*dt*k1_w)
    k2_w = omega_dot(omega + 0.5*dt*k1_w, J)

    k3_q = quat_dot(q + 0.5*dt*k2_q, omega + 0.5*dt*k2_w)
    k3_w = omega_dot(omega + 0.5*dt*k2_w, J)

    k4_q = quat_dot(q + dt*k3_q, omega + dt*k3_w)
    k4_w = omega_dot(omega + dt*k3_w, J)

    q_next = q + (dt/6)*(k1_q + 2*k2_q + 2*k3_q + k4_q)
    omega_next = omega + (dt/6)*(k1_w + 2*k2_w + 2*k3_w + k4_w)

    return quat_normalize(q_next), omega_next

Dynamics

In [ ]:
def omega_dot(omega, J):
    return np.linalg.inv(J) @ (-np.cross(omega, J @ omega))

def quat_dot(q, omega):
    wx, wy, wz = omega
    Omega = np.array([
        [0, -wx, -wy, -wz],
        [wx, 0, wz, -wy],
        [wy, -wz, 0, wx],
        [wz, wy, -wx, 0]
    ])
    return 0.5 * Omega @ q

def rk4_step(q, omega, dt, J):
    k1_q = quat_dot(q, omega)
    k1_w = omega_dot(omega, J)

    k2_q = quat_dot(q + 0.5*dt*k1_q, omega + 0.5*dt*k1_w)
    k2_w = omega_dot(omega + 0.5*dt*k1_w, J)

    k3_q = quat_dot(q + 0.5*dt*k2_q, omega + 0.5*dt*k2_w)
    k3_w = omega_dot(omega + 0.5*dt*k2_w, J)

    k4_q = quat_dot(q + dt*k3_q, omega + dt*k3_w)
    k4_w = omega_dot(omega + dt*k3_w, J)

    q_next = q + (dt/6)*(k1_q + 2*k2_q + 2*k3_q + k4_q)
    omega_next = omega + (dt/6)*(k1_w + 2*k2_w + 2*k3_w + k4_w)

    return quat_normalize(q_next), omega_next

SENSOR MODELS (vector, star tracker & gyroscope)

In [ ]:
def vector_sensor(v_inertial, q):
    R = quat_to_dcm(q)
    v_body_true = R.T @ v_inertial

    M = np.eye(3) + 0.01*np.random.randn(3,3)
    b = np.array([1e-3, -1e-3, 2e-3])
    W = (1e-4)**2 * np.eye(3)

    w = np.random.multivariate_normal(np.zeros(3), W)

    return v_body_true, M @ v_body_true + b + w

In [ ]:
def star_tracker(q_true):
    sigma_xy = np.deg2rad(0.005)
    sigma_z  = np.deg2rad(0.05)

    cov = np.diag([sigma_xy**2, sigma_xy**2, sigma_z**2])
    delta_theta = np.random.multivariate_normal(np.zeros(3), cov)

    dq = small_angle_quat(delta_theta)
    return quat_multiply(dq, q_true)

In [ ]:
class Gyro:
    def __init__(self):
        self.b = np.zeros(3)

    def measure(self, omega_true, dt):
        M = np.eye(3) + 0.001*np.random.randn(3,3)

        Qb = (1e-5)**2 * np.eye(3)
        self.b += np.random.multivariate_normal(np.zeros(3), Qb*dt)

        W = (1e-3)**2 * np.eye(3)
        w = np.random.multivariate_normal(np.zeros(3), W)

        return M @ omega_true + self.b + w

Simulation

In [ ]:
dt = 0.1
T = 200
N = int(T/dt)

J = np.diag([0.0867, 0.1817, 0.1117])

q = np.array([1,0,0,0])
omega = np.array([0.1, 0.2, 0.15])

gyro = Gyro()

v_inertial = np.array([1,0,0])

omega_hist = []
gyro_hist = []
vec_true = []
vec_meas = []
star_err = []

def quat_error(q_true, q_meas):
    dq = quat_multiply(q_meas, np.array([q_true[0], -q_true[1], -q_true[2], -q_true[3]]))
    return np.linalg.norm(dq[1:])

Run

In [ ]:
for i in range(N):
    omega_hist.append(omega)

    gyro_m = gyro.measure(omega, dt)
    gyro_hist.append(gyro_m)

    v_t, v_m = vector_sensor(v_inertial, q)
    vec_true.append(v_t)
    vec_meas.append(v_m)

    q_star = star_tracker(q)
    star_err.append(quat_error(q, q_star))

    q, omega = rk4_step(q, omega, dt, J)

PLOTS

In [ ]:
plt.figure()
plt.plot(omega_hist, label="true")
plt.plot(gyro_hist, linestyle='--', alpha=0.7)
plt.title("Gyroscope Measurement vs Truth")
plt.savefig("figures/gyro.png", dpi=300)
plt.show()

In [ ]:
vec_true = np.array(vec_true)
vec_meas = np.array(vec_meas)

plt.figure()
plt.plot(vec_true[:,0], label="true")
plt.plot(vec_meas[:,0], linestyle='--', label="measured")
plt.legend()
plt.title("Vector Sensor Measurement")
plt.savefig("figures/vector.png", dpi=300)
plt.show()

In [ ]:
plt.figure()
plt.plot(star_err)
plt.title("Star Tracker Error Magnitude")
plt.savefig("figures/star.png", dpi=300)
plt.show()